# 06. Residual routing — residual, mHC, and depth attention

이 노트북에서는 모델 크기만 줄인다. residual routing의 계산 그래프는 줄이지 않는다.

- ordinary residual / LayerScale
- mHC의 `H_pre`, `H_res`, `H_post`
- `H_res`의 Sinkhorn projection
- residual streams를 여러 Transformer layer 동안 유지하는 persistent mHC
- Kimi K3 계열의 Block Attention Residuals: depth 방향으로 이전 block representation을 선택적으로 읽기


In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(7)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)


## 1. Residual and LayerScale


In [ ]:
x = torch.randn(2, 5, 24, device=device)
branch = nn.Sequential(
    nn.Linear(24, 48),
    nn.SiLU(),
    nn.Linear(48, 24),
).to(device)

gamma = nn.Parameter(1e-4 * torch.ones(24, device=device))
plain = x + branch(x)
scaled = x + gamma * branch(x)

print("plain:", plain.shape)
print("LayerScale initial change:", (scaled - x).norm().item())


## 2. mHC routing maps

Residual state is `[batch, tokens, streams, channels]`.
For every branch, mHC first reads multiple streams with `H_pre`,
mixes the residual streams with a doubly-stochastic `H_res`,
then writes the branch result back with `H_post`.


In [ ]:
def sinkhorn(logits, iterations=20):
    matrix = logits.float().exp()
    for _ in range(iterations):
        matrix = matrix / matrix.sum(dim=-1, keepdim=True)
        matrix = matrix / matrix.sum(dim=-2, keepdim=True)
    return matrix.to(logits.dtype)


class DynamicMHCRouter(nn.Module):
    def __init__(self, streams=4, channels=24, dynamic_scale=1e-2):
        super().__init__()
        self.streams = streams
        self.norm = nn.RMSNorm(channels)

        self.pre = nn.Linear(channels, streams, bias=False)
        self.post = nn.Linear(channels, streams, bias=False)
        self.res = nn.Linear(channels, streams * streams, bias=False)

        self.alpha_pre = nn.Parameter(torch.tensor(dynamic_scale))
        self.alpha_post = nn.Parameter(torch.tensor(dynamic_scale))
        self.alpha_res = nn.Parameter(torch.tensor(dynamic_scale))

        self.base_pre = nn.Parameter(torch.ones(streams))
        self.base_post = nn.Parameter(torch.zeros(streams))
        self.base_res = nn.Parameter(4.0 * torch.eye(streams))

    def forward(self, streams):
        summary = self.norm(streams.mean(dim=2))
        batch, tokens, _ = summary.shape

        raw_pre = self.base_pre + self.alpha_pre * self.pre(summary)
        raw_post = self.base_post + self.alpha_post * self.post(summary)

        dynamic_res = self.res(summary).view(
            batch, tokens, self.streams, self.streams
        )
        raw_res = self.base_res + self.alpha_res * dynamic_res

        h_pre = torch.sigmoid(raw_pre)
        h_post = 2.0 * torch.sigmoid(raw_post)
        h_res = sinkhorn(raw_res)
        return h_pre, h_post, h_res


## 3. One complete mHC branch


In [ ]:
class MHCBranch(nn.Module):
    def __init__(self, branch, streams=4, channels=24):
        super().__init__()
        self.router = DynamicMHCRouter(streams, channels)
        self.branch = branch

    def forward(self, residual_streams):
        h_pre, h_post, h_res = self.router(residual_streams)

        branch_input = torch.einsum(
            "bts,btsd->btd",
            h_pre,
            residual_streams,
        )
        branch_output = self.branch(branch_input)

        residual_mixed = torch.einsum(
            "btsr,btrd->btsd",
            h_res,
            residual_streams,
        )
        branch_written = h_post[..., None] * branch_output[:, :, None, :]

        return residual_mixed + branch_written, (h_pre, h_post, h_res)


## 4. Persistent mHC Transformer

The expanded streams are **not collapsed after one demonstration**.
Attention and FFN each receive their own mHC routing and the resulting streams
are propagated through all layers. Only the final readout collapses streams.


In [ ]:
class MHCAttention(nn.Module):
    def __init__(self, channels=24, heads=3):
        super().__init__()
        self.norm = nn.RMSNorm(channels)
        self.attn = nn.MultiheadAttention(
            channels,
            heads,
            batch_first=True,
        )

    def forward(self, x):
        h = self.norm(x)
        h, _ = self.attn(h, h, h, need_weights=False)
        return h


class MHCFFN(nn.Module):
    def __init__(self, channels=24):
        super().__init__()
        self.norm = nn.RMSNorm(channels)
        self.ffn = nn.Sequential(
            nn.Linear(channels, 4 * channels),
            nn.SiLU(),
            nn.Linear(4 * channels, channels),
        )

    def forward(self, x):
        return self.ffn(self.norm(x))


class MHCTransformerLayer(nn.Module):
    def __init__(self, streams=4, channels=24, heads=3):
        super().__init__()
        self.attention = MHCBranch(
            MHCAttention(channels, heads),
            streams,
            channels,
        )
        self.ffn = MHCBranch(
            MHCFFN(channels),
            streams,
            channels,
        )

    def forward(self, residual_streams):
        residual_streams, attn_maps = self.attention(residual_streams)
        residual_streams, ffn_maps = self.ffn(residual_streams)
        return residual_streams, (attn_maps, ffn_maps)


class PersistentMHCTransformer(nn.Module):
    def __init__(self, depth=3, streams=4, channels=24, heads=3):
        super().__init__()
        self.streams = streams
        self.expand = nn.Linear(channels, streams * channels)
        self.layers = nn.ModuleList(
            [
                MHCTransformerLayer(streams, channels, heads)
                for _ in range(depth)
            ]
        )
        self.readout_logits = nn.Parameter(torch.zeros(streams))

    def forward(self, x):
        batch, tokens, channels = x.shape
        residual_streams = self.expand(x).view(
            batch,
            tokens,
            self.streams,
            channels,
        )

        routing_history = []
        for layer in self.layers:
            residual_streams, maps = layer(residual_streams)
            routing_history.append(maps)

        readout = self.readout_logits.softmax(dim=0)
        output = torch.einsum(
            "s,btsd->btd",
            readout,
            residual_streams,
        )
        return output, residual_streams, routing_history


mhc_model = PersistentMHCTransformer().to(device)
mhc_input = torch.randn(2, 7, 24, device=device)
mhc_output, final_streams, history = mhc_model(mhc_input)

loss = mhc_output.square().mean()
loss.backward()

last_h_res = history[-1][1][2]
print("output:", mhc_output.shape)
print("persistent streams:", final_streams.shape)
print("H_res row sums:", last_h_res[0, 0].sum(dim=-1))
print("H_res col sums:", last_h_res[0, 0].sum(dim=-2))
print("first layer router grad:",
      mhc_model.layers[0].attention.router.res.weight.grad.norm().item())


## 5. Block Attention Residuals

Kimi K3 uses depth attention rather than blindly summing every previous layer.
For a small executable version, two ordinary residual layers form one block.
Completed blocks are retained as depth sources. At a new block boundary,
a learned pseudo-query scores the normalized completed-block representations
and retrieves a weighted mixture before the next block starts.

The official model uses a larger block size; here only the **depth dimension**
is reduced.


In [ ]:
class BlockAttentionResidual(nn.Module):
    def __init__(self, channels=24):
        super().__init__()
        self.query = nn.Parameter(torch.randn(channels) * 0.02)
        self.norm = nn.RMSNorm(channels)

    def retrieve(self, block_sources):
        stacked = torch.stack(block_sources, dim=2)
        normalized = self.norm(stacked)

        scores = torch.einsum(
            "d,btkd->btk",
            self.query,
            normalized,
        ) / math.sqrt(normalized.size(-1))

        weights = scores.softmax(dim=-1)
        retrieved = torch.einsum(
            "btk,btkd->btd",
            weights,
            stacked,
        )
        return retrieved, weights


class DepthResidualLayer(nn.Module):
    def __init__(self, channels=24):
        super().__init__()
        self.norm = nn.RMSNorm(channels)
        self.ffn = nn.Sequential(
            nn.Linear(channels, 4 * channels),
            nn.GELU(),
            nn.Linear(4 * channels, channels),
        )

    def forward(self, x):
        return x + self.ffn(self.norm(x))


class BlockAttnResStack(nn.Module):
    def __init__(self, depth=6, block_size=2, channels=24):
        super().__init__()
        self.block_size = block_size
        self.layers = nn.ModuleList(
            [DepthResidualLayer(channels) for _ in range(depth)]
        )
        self.depth_attention = BlockAttentionResidual(channels)

    def forward(self, x):
        block_sources = [x]
        depth_weights = []

        for layer_index, layer in enumerate(self.layers):
            if layer_index > 0 and layer_index % self.block_size == 0:
                x, weights = self.depth_attention.retrieve(block_sources)
                depth_weights.append(weights)

            x = layer(x)

            if (layer_index + 1) % self.block_size == 0:
                block_sources.append(x)

        return x, depth_weights


attnres = BlockAttnResStack().to(device)
attnres_output, depth_weights = attnres(
    torch.randn(2, 7, 24, device=device)
)
print("AttnRes output:", attnres_output.shape)
print("depth-attention sites:", len(depth_weights))
print("last depth weights:", depth_weights[-1][0, 0])


## References and provenance

- **mHC (Manifold-Constrained Hyper-Connections)**: expanded residual streams,
  input-dependent `H_pre/H_post/H_res`, and Sinkhorn projection of `H_res`.
- **Kimi K3 Attention Residuals**: information is selectively retrieved across
  model depth. The miniature implementation keeps block-level depth sources
  and reduces only depth/block size.
